# IMD Zone RMSE Ranking and Long-Term Trend Analysis

Detailed RMSE ranking across IMD meteorological zones and extraction of long-term
warming trends and inter-annual variability statistics from the DDPM downscaled outputs.

**Before running:** Update all path variables marked `<SET YOUR PATH HERE>` to your local data directories.

**Paper reference:** Jha, S.K., Gupta, V., et al. (2026). A Generative AI Framework for High-Fidelity Probabilistic Downscaling of CMIP6 Daily Temperature Projections over India. *Communications Earth & Environment.*

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# ===================================================================
# 1. CONFIGURATION
# ===================================================================

datasets_paths = {
    "MSWX": "<SET YOUR PATH HERE>",
    "IMDAA": "<SET YOUR PATH HERE>",
    "ERA5": "<SET YOUR PATH HERE>",
    "IMD": "<SET YOUR PATH HERE>"
}

varnames = {
    "MSWX": "air_temperature",
    "IMDAA": "TMP_2m",
    "ERA5": "t2m",
    "IMD": "tmean"
}

DATASET_ORDER = ["IMD", "MSWX", "ERA5", "IMDAA"]
CLIM_PERIOD = slice("1985", "2014")

# ===================================================================
# 2. STATISTICAL FUNCTIONS
# ===================================================================

def detect_and_convert_units(da):
    if "units" in da.attrs and "K" in da.attrs["units"]:
        return da - 273.15
    if da.mean().compute() > 200:
        return da - 273.15
    return da

def spatial_stats(map_da, label):
    vals = map_da.values
    vals = vals[np.isfinite(vals)]
    return {
        "Dataset": label,
        "Mean": np.mean(vals),
        "Median": np.median(vals),
        "Std": np.std(vals),
        "Min": np.min(vals),
        "Max": np.max(vals),
        "P05": np.percentile(vals, 5),
        "P95": np.percentile(vals, 95),
    }

def climate_stats(da, label):
    vals = da.values.flatten()
    vals = vals[np.isfinite(vals)]
    return {
        "Dataset": label,
        "Mean": np.mean(vals),
        "Median": np.median(vals),
        "Std": np.std(vals),
        "Min": np.min(vals),
        "Max": np.max(vals),
        "P05": np.percentile(vals, 5),
        "P95": np.percentile(vals, 95),
    }

def _align_and_mask(map1, map2):
    m1, m2 = xr.align(map1, map2, join="inner")
    f1, f2 = m1.values.flatten(), m2.values.flatten()
    mask = np.isfinite(f1) & np.isfinite(f2)
    return f1[mask], f2[mask]

def comparative_metrics(model, ref):
    m, r = _align_and_mask(model, ref)
    corr, _ = pearsonr(m, r)
    diff = m - r
    return {
        "SPC": corr,
        "RMSE": np.sqrt(np.mean(diff**2)),
        "Bias": np.mean(diff),
    }

# ===================================================================
# 3. LOAD DATA
# ===================================================================

data_store = {}

for name, path in datasets_paths.items():
    try:
        ds = xr.open_dataset(path, chunks={"time": 365})
        if "latitude" in ds.dims: ds = ds.rename({"latitude": "lat"})
        if "longitude" in ds.dims: ds = ds.rename({"longitude": "lon"})
        da = ds[varnames[name]].sel(time=CLIM_PERIOD)
        da = detect_and_convert_units(da)
        data_store[name] = da
    except Exception as e:
        print(f"{name} failed: {e}")

# ===================================================================
# 4. AGGREGATES
# ===================================================================

aggregates = {}

for name in DATASET_ORDER:
    da = data_store[name]
    ann = da.mean("time").compute()
    may = da.sel(time=da.time.dt.month == 5).mean("time").compute()
    aggregates[name] = {"ann": ann, "may": may}

# ===================================================================
# 5. STATISTICS
# ===================================================================

spatial_ann, spatial_may = [], []
climate_ann, climate_may = [], []

for name in DATASET_ORDER:
    spatial_ann.append(spatial_stats(aggregates[name]["ann"], name))
    spatial_may.append(spatial_stats(aggregates[name]["may"], name))
    climate_ann.append(climate_stats(data_store[name], name))
    climate_may.append(climate_stats(
        data_store[name].sel(time=data_store[name].time.dt.month == 5), name))

# ===================================================================
# 6. COMPARATIVE METRICS
# ===================================================================

ref_ann = aggregates["IMD"]["ann"]
ref_may = aggregates["IMD"]["may"]

comp_ann, comp_may = [], []

for name in DATASET_ORDER:
    if name == "IMD": continue
    comp_ann.append({"Dataset": name, **comparative_metrics(aggregates[name]["ann"], ref_ann)})
    comp_may.append({"Dataset": name, **comparative_metrics(aggregates[name]["may"], ref_may)})

# ===================================================================
# 7. OUTPUT
# ===================================================================

print("\nSPATIAL CLIMATOLOGY (grid-cell means)")
print(pd.DataFrame(spatial_ann).round(2).to_markdown(index=False))

print("\nTRUE CLIMATE DISTRIBUTION (time × space)")
print(pd.DataFrame(climate_ann).round(2).to_markdown(index=False))

print("\nSPATIAL PERFORMANCE (Annual vs IMD)")
print(pd.DataFrame(comp_ann).round(3).to_markdown(index=False))

print("\nSPATIAL PERFORMANCE (May vs IMD)")
print(pd.DataFrame(comp_may).round(3).to_markdown(index=False))


In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
import rioxarray
import geopandas as gpd
import warnings

# Suppress warnings for cleaner text output
warnings.filterwarnings("ignore")

# -----------------------------
# 1. Paths (Preserved from original)
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"

# -----------------------------
# 2. Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365): 
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

def calculate_distribution_stats(da, metric_name):
    """Calculates spatial distribution statistics for a 2D DataArray."""
    vals = da.values.flatten()
    vals = vals[np.isfinite(vals)] # Remove NaNs (masked areas)
    
    if len(vals) == 0:
        return {k: np.nan for k in ['Metric', 'Mean', 'Median', 'Std', 'Min', 'Max', 'P05', 'P95']}

    return {
        'Metric': metric_name,
        'Mean': np.mean(vals),
        'Median': np.median(vals),
        'Std': np.std(vals),
        'Min': np.min(vals),
        'Max': np.max(vals),
        'P05': np.percentile(vals, 5),
        'P25': np.percentile(vals, 25),
        'P75': np.percentile(vals, 75),
        'P95': np.percentile(vals, 95)
    }

# -----------------------------
# 3. Load & Process Data
# -----------------------------
print("Initializing Data Processing...")

# Find files
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

# Filter period
f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"  Files found: {len(f_mswx)} MSWX, {len(f_ddpm)} DDPM")

# Open Datasets
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# Assign coordinates
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# -----------------------------
# 4. Calculation: Means & Biases
# -----------------------------
print("  Calculating Temporal Means (Climatology)...")
da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

# Annual & Overall Means
annual_mswx = da_m.groupby("time.year").mean("time")
annual_ddpm = da_d.groupby("time.year").mean("time")

overall_mswx = annual_mswx.mean("year")
overall_ddpm = annual_ddpm.mean("year")
overall_bias = overall_ddpm - overall_mswx # Bias of the mean fields

# -----------------------------
# 5. Calculation: Time-Series Metrics
# -----------------------------
print("  Calculating Time-Series Metrics (RMSE, MAE, Corr)...")

# Bias time series (for RMSE/MAE calculation)
bias_ts = da_d - da_m

# Temporal RMSE Map (Root Mean Square of the bias time series at each pixel)
rmse_map = np.sqrt((bias_ts ** 2).mean("time", skipna=True))

# Temporal MAE Map
mae_map = np.abs(bias_ts).mean("time", skipna=True)

# Temporal Correlation Map
corr_map = xr.corr(da_d, da_m, dim="time")

# -----------------------------
# 6. Spatial Clipping
# -----------------------------
print("  Clipping to Shapefile...")
gdf = gpd.read_file(india_shp)

data_maps = {
    "MSWX_Mean": overall_mswx,
    "DDPM_Mean": overall_ddpm,
    "Bias_Mean": overall_bias,
    "RMSE_Temporal": rmse_map,
    "MAE_Temporal": mae_map,
    "Correlation": corr_map
}

clipped_maps = {}

for name, da in data_maps.items():
    # Rename lat/lon if needed
    if 'lat' in da.dims: da = da.rename({"lat": "y", "lon": "x"})
    
    # Set CRS
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)
    
    # Clip
    clipped_maps[name] = da.rio.clip(gdf.geometry, gdf.crs)

# -----------------------------
# 7. Generate Statistics Table
# -----------------------------
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT")
print("="*80 + "\n")

print(f"**Dataset Analysis: DDPM Downscaling Results**")
print(f"Period: 2009-08-07 to 2014-12-31 (n={len(f_mswx)} days)")
print("Target: MSWX | Prediction: DDPM")

# --- Table 1: Climatology (Absolute Values) ---
print("\n### 1. Climatological Fields (Spatial Statistics of Time-Mean Maps)")
stats_clim = []
stats_clim.append(calculate_distribution_stats(clipped_maps["MSWX_Mean"], "MSWX Mean Temp (°C)"))
stats_clim.append(calculate_distribution_stats(clipped_maps["DDPM_Mean"], "DDPM Mean Temp (°C)"))
df_clim = pd.DataFrame(stats_clim).round(2)
print(df_clim.to_markdown(index=False))

# --- Table 2: Error Analysis ---
print("\n### 2. Error Analysis (Spatial Statistics of Error Maps)")
stats_err = []
stats_err.append(calculate_distribution_stats(clipped_maps["Bias_Mean"], "Mean Bias (MB) (°C)"))
stats_err.append(calculate_distribution_stats(clipped_maps["MAE_Temporal"], "Temporal MAE (°C)"))
stats_err.append(calculate_distribution_stats(clipped_maps["RMSE_Temporal"], "Temporal RMSE (°C)"))
df_err = pd.DataFrame(stats_err).round(3)
print(df_err.to_markdown(index=False))

# --- Table 3: Correlation Analysis ---
print("\n### 3. Correlation Analysis (Spatial Statistics of Pearson r Map)")
stats_corr = []
stats_corr.append(calculate_distribution_stats(clipped_maps["Correlation"], "Temporal Correlation (r)"))
df_corr = pd.DataFrame(stats_corr).round(3)
print(df_corr.to_markdown(index=False))

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import os
import warnings
from scipy.stats import pearsonr

# Suppress warnings
warnings.filterwarnings("ignore")

# ===================================================================
# 1. CONFIGURATION
# ===================================================================

# --- Paths (Adjusted from your previous codes) ---
# High Res Target (MSWX)
path_hr = "<SET YOUR PATH HERE>"
# Super Res Output (DDPM)
path_sr = "<SET YOUR PATH HERE>"
# Low Res Input (ERA5 - using path from Code 1)
path_lr = "<SET YOUR PATH HERE>"

# --- Output Settings ---
# Transect coordinates (Longitude fixed, varying Latitude for Himalayas)
TRANSECT_LON = 77.0  # Approx longitude for Mandi/Himachal
TRANSECT_LAT_RANGE = slice(28, 36) 

# ===================================================================
# 2. HELPER FUNCTIONS
# ===================================================================

def get_stats(da, case_name, dataset_name):
    """Calculates spatial statistics for a specific map."""
    vals = da.values.flatten()
    vals = vals[np.isfinite(vals)]
    
    if len(vals) == 0:
        return None

    return {
        'Case Study': case_name,
        'Dataset': dataset_name,  # <--- FIXED: Explicitly added this key
        'Mean': np.mean(vals),
        'Std': np.std(vals),
        'Min': np.min(vals),
        'P05': np.percentile(vals, 5),
        'P95': np.percentile(vals, 95),
        'Max': np.max(vals)
    }

def get_perf_metrics(da_pred, da_ref, case_name, dataset_name):
    """Calculates RMSE, Bias, and SPC against reference."""
    # Align and mask
    a, b = xr.align(da_pred, da_ref, join="inner")
    a_flat = a.values.flatten()
    b_flat = b.values.flatten()
    
    mask = np.isfinite(a_flat) & np.isfinite(b_flat)
    a_valid = a_flat[mask]
    b_valid = b_flat[mask]
    
    if len(a_valid) < 10:
        return None

    diff = a_valid - b_valid
    rmse = np.sqrt(np.mean(diff**2))
    bias = np.mean(diff)
    corr, _ = pearsonr(a_valid, b_valid)

    return {
        'Case Study': case_name,
        'Dataset': dataset_name,
        'RMSE': rmse,
        'Bias': bias,
        'SPC': corr # Spatial Pattern Correlation
    }

# ===================================================================
# 3. DATA LOADING
# ===================================================================
print("Loading Datasets...")

# Load LR (ERA5)
try:
    ds_lr = xr.open_dataset(path_lr)
    if 't2m' in ds_lr: ds_lr = ds_lr.rename({'t2m': 'air_temperature'})
    # Unit check
    if ds_lr['air_temperature'].mean().compute().item() > 200:
        ds_lr['air_temperature'] -= 273.15
except Exception as e:
    print(f"Error loading LR: {e}")
    ds_lr = None

# Load HR (MSWX) & SR (DDPM)
# Using 'open_mfdataset' assuming they are split files, or standard open if single
try:
    # Attempt simple open first for speed if they are folders of NCs
    if os.path.isdir(path_hr):
        ds_hr = xr.open_mfdataset(f"{path_hr}/*.nc", combine='nested', concat_dim='time')
    else:
        ds_hr = xr.open_dataset(path_hr)
        
    if os.path.isdir(path_sr):
        ds_sr = xr.open_mfdataset(f"{path_sr}/*.nc", combine='nested', concat_dim='time')
    else:
        ds_sr = xr.open_dataset(path_sr)

    # Unit Conversion if needed
    if 'time' not in ds_hr.coords:
        # If time is missing, you might need the date parsing logic from Code 2
        # For this script, assuming 'time' exists or is added.
        print("Warning: 'time' coordinate checking needed.")

except Exception as e:
    print(f"Error loading HR/SR: {e}")
    ds_hr = None
    ds_sr = None

# Ensure common time intersection for safety
common_time = np.intersect1d(ds_hr.time, ds_sr.time)
# Also intersect with LR if possible, or select nearest
print(f"Common time steps found: {len(common_time)}")

# ===================================================================
# 4. CASE STUDY EXTRACTION
# ===================================================================
stats_registry = []
perf_registry = []

# --- Select Dates ---
# 1. Hot Day: Find Max Mean Temp in HR
t_mean_series = ds_hr['air_temperature'].mean(dim=['lat', 'lon'])
hot_date = t_mean_series.idxmax().values
print(f"Identified Hot Day: {pd.to_datetime(hot_date).strftime('%Y-%m-%d')}")

# 2. Cold Day: Find Min Mean Temp in HR
cold_date = t_mean_series.idxmin().values
print(f"Identified Cold Day: {pd.to_datetime(cold_date).strftime('%Y-%m-%d')}")

# 3. Monsoon: July 2011 (arbitrary valid monsoon month in your range)
monsoon_year = pd.to_datetime(common_time[0]).year + 1 # Pick a year in the middle
monsoon_start = f"{monsoon_year}-07-01"
monsoon_end = f"{monsoon_year}-07-31"
print(f"Identified Monsoon Period: July {monsoon_year}")

cases = {
    "Hot Day": {'method': 'sel', 'args': {'time': hot_date}},
    "Cold Day": {'method': 'sel', 'args': {'time': cold_date}},
    "Monsoon (July)": {'method': 'sel_mean', 'slice': slice(monsoon_start, monsoon_end)}
}

# --- Loop Over Cases ---
for case_name, params in cases.items():
    print(f"Processing: {case_name}")
    
    data_maps = {}
    
    # Extract Maps
    for name, ds in [('Low Res', ds_lr), ('High Res', ds_hr), ('DDPM', ds_sr)]:
        if ds is None: continue
        
        try:
            if params['method'] == 'sel':
                # Select specific date (nearest for LR to handle mismatched times)
                da = ds['air_temperature'].sel(time=params['args']['time'], method='nearest')
            elif params['method'] == 'sel_mean':
                # Average over a period
                da = ds['air_temperature'].sel(time=params['slice']).mean('time')
            
            data_maps[name] = da
        except Exception as e:
            print(f"  Skipping {name} for {case_name}: {e}")

    # Calculate Stats
    ref_map = data_maps.get('High Res')
    
    for name, da in data_maps.items():
        # 1. Distribution Stats
        s = get_stats(da, case_name, name)
        if s: stats_registry.append(s)
        
        # 2. Performance Stats (vs High Res)
        if name != 'High Res' and ref_map is not None:
            p = get_perf_metrics(da, ref_map, case_name, name)
            if p: perf_registry.append(p)

# ===================================================================
# 5. TRANSECT ANALYSIS (Himalayan Cross-Section)
# ===================================================================
print("Processing: Himalayan Transect")
# Use the "Monsoon" maps for the transect
try:
    case_name = "Transect (Monsoon)"
    
    # Get Monsoon maps again
    monsoon_slice = slice(monsoon_start, monsoon_end)
    
    # Extract profiles along Longitude
    # We select a specific Longitude and a Range of Latitudes
    prof_hr = ds_hr['air_temperature'].sel(time=monsoon_slice).mean('time').sel(lon=TRANSECT_LON, method='nearest').sel(lat=TRANSECT_LAT_RANGE)
    prof_sr = ds_sr['air_temperature'].sel(time=monsoon_slice).mean('time').sel(lon=TRANSECT_LON, method='nearest').sel(lat=TRANSECT_LAT_RANGE)
    prof_lr = ds_lr['air_temperature'].sel(time=monsoon_slice).mean('time').sel(lon=TRANSECT_LON, method='nearest').sel(lat=TRANSECT_LAT_RANGE)

    # Calculate Profile Correlations (1D)
    # Align
    def calc_profile_metrics(p_model, p_ref, name):
        a, b = xr.align(p_model, p_ref)
        corr, _ = pearsonr(a.values, b.values)
        rmse = np.sqrt(np.mean((a.values - b.values)**2))
        return {
            'Case Study': case_name,
            'Dataset': name,
            'RMSE': rmse,
            'Bias': np.mean(a.values - b.values),
            'SPC': corr
        }

    perf_registry.append(calc_profile_metrics(prof_lr, prof_hr, "Low Res"))
    perf_registry.append(calc_profile_metrics(prof_sr, prof_hr, "DDPM"))

except Exception as e:
    print(f"Transect Analysis failed: {e}")


# ===================================================================
# 6. OUTPUT GENERATION
# ===================================================================
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT")
print("="*80 + "\n")

print("**Case Study Analysis: Extreme Events & Regional Transects**")
print("Comparison: Low-Res (Input) vs. DDPM (Prediction) vs. High-Res (Target)")

# --- Table 1: Spatial Statistics ---
print("\n### 1. Temperature Distributions (Spatial Statistics)")
df_dist = pd.DataFrame(stats_registry)
# Reorder columns for readability
cols_dist = ['Case Study', 'Dataset', 'Mean', 'Std', 'Min', 'P05', 'P95', 'Max']
# Filter columns that exist
cols_dist = [c for c in cols_dist if c in df_dist.columns]
print(df_dist[cols_dist].round(2).to_markdown(index=False))

# --- Table 2: Performance Comparison ---
print("\n### 2. Performance Metrics (Relative to High-Res Target)")
df_perf = pd.DataFrame(perf_registry)
cols_perf = ['Case Study', 'Dataset', 'RMSE', 'Bias', 'SPC']
# Filter columns
cols_perf = [c for c in cols_perf if c in df_perf.columns]
print(df_perf[cols_perf].round(3).to_markdown(index=False))

print("\n### 3. Transect Interpretation")
print(f"Transect taken at Longitude {TRANSECT_LON}E (Himalayan Region).")
print("Metrics indicate ability to capture orographic gradients.")

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
import rioxarray
import geopandas as gpd
import warnings

# Suppress warnings for cleaner text output
warnings.filterwarnings("ignore")

# ===================================================================
# 1. CONFIGURATION
# ===================================================================
# Paths (Preserved from your code)
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"

# ===================================================================
# 2. HELPER FUNCTIONS
# ===================================================================
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365): 
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

def get_spatial_distribution(da, metric_name):
    """
    Calculates detailed spatial statistics for a 2D map.
    This replaces the visual plot with numbers representing the distribution.
    """
    vals = da.values.flatten()
    vals = vals[np.isfinite(vals)] # Remove NaNs
    
    if len(vals) == 0:
        return {k: np.nan for k in ['Metric', 'Mean', 'Std', 'Min', 'P05', 'P25', 'Median', 'P75', 'P95', 'Max']}

    return {
        'Metric': metric_name,
        'Mean': np.mean(vals),       # Average value across India
        'Std': np.std(vals),         # Spatial variability
        'Min': np.min(vals),
        'P05': np.percentile(vals, 5),   # Lower extreme
        'P25': np.percentile(vals, 25),
        'Median': np.median(vals),
        'P75': np.percentile(vals, 75),
        'P95': np.percentile(vals, 95),  # Upper extreme
        'Max': np.max(vals)
    }

# ===================================================================
# 3. DATA PROCESSING
# ===================================================================
print("Initializing Data Processing...")

# --- Load Files ---
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"  Files found: {len(f_mswx)} MSWX, {len(f_ddpm)} DDPM")

if len(f_mswx) == 0 or len(f_ddpm) == 0:
    print("CRITICAL: No files found. Check paths and dates.")
    exit()

# --- Open Datasets ---
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# --- Assign Time ---
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# --- Calculate Climatology (Time Means) ---
print("  Calculating Climatological Means...")
da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

# Annual & Overall Means
annual_mswx = da_m.groupby("time.year").mean("time")
annual_ddpm = da_d.groupby("time.year").mean("time")

overall_mswx = annual_mswx.mean("year")
overall_ddpm = annual_ddpm.mean("year")
overall_bias = overall_ddpm - overall_mswx # The "Bias Map"

# --- Calculate Temporal Error Metrics (RMSE/MAE/Corr Maps) ---
print("  Calculating Temporal Error Maps (RMSE, MAE, Corr)...")

# Bias time series (Prediction - Target at every step)
bias_ts = da_d - da_m

# Temporal RMSE Map: sqrt(mean(bias^2)) along time
rmse_map = np.sqrt((bias_ts ** 2).mean("time", skipna=True))

# Temporal MAE Map: mean(abs(bias)) along time
mae_map = np.abs(bias_ts).mean("time", skipna=True)

# Temporal Correlation Map: Pearson r along time
corr_map = xr.corr(da_d, da_m, dim="time")

# --- Spatial Clipping ---
print("  Clipping to Shapefile...")
gdf = gpd.read_file(india_shp)

maps_to_process = {
    "MSWX Mean (Climatology)": overall_mswx,
    "DDPM Mean (Climatology)": overall_ddpm,
    "Bias (Climatology)": overall_bias,
    "Temporal RMSE": rmse_map,
    "Temporal MAE": mae_map,
    "Temporal Correlation": corr_map
}

clipped_results = {}

for name, da in maps_to_process.items():
    # Rename lat/lon to y/x for rioxarray
    if 'lat' in da.dims: da = da.rename({"lat": "y", "lon": "x"})
    
    # Set CRS
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)
    
    # Clip
    try:
        clipped = da.rio.clip(gdf.geometry, gdf.crs)
        clipped_results[name] = clipped
    except Exception as e:
        print(f"Warning: Clipping failed for {name}: {e}")

# ===================================================================
# 4. OUTPUT GENERATION
# ===================================================================
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
print("="*80 + "\n")

print(f"**Analysis of DDPM Spatial Performance vs MSWX**")
print(f"Observation Period: 2009-08-07 to 2014-12-31")
print(f"Region: India (Clipped)")

# --- Table 1: Climatological Comparison ---
print("\n### 1. Climatological Statistics (Spatial Distribution of Mean Temperature)")
stats_clim = []
if "MSWX Mean (Climatology)" in clipped_results:
    stats_clim.append(get_spatial_distribution(clipped_results["MSWX Mean (Climatology)"], "MSWX Mean (°C)"))
if "DDPM Mean (Climatology)" in clipped_results:
    stats_clim.append(get_spatial_distribution(clipped_results["DDPM Mean (Climatology)"], "DDPM Mean (°C)"))

if stats_clim:
    df_clim = pd.DataFrame(stats_clim).round(2)
    # Reorder columns
    cols = ['Metric', 'Mean', 'Median', 'Std', 'Min', 'P05', 'P95', 'Max']
    print(df_clim[cols].to_markdown(index=False))

# --- Table 2: Error Analysis ---
print("\n### 2. Error Analysis (Spatial Distribution of Error Metrics)")
stats_err = []
if "Bias (Climatology)" in clipped_results:
    stats_err.append(get_spatial_distribution(clipped_results["Bias (Climatology)"], "Mean Bias (MB) (°C)"))
if "Temporal MAE" in clipped_results:
    stats_err.append(get_spatial_distribution(clipped_results["Temporal MAE"], "Temporal MAE (°C)"))
if "Temporal RMSE" in clipped_results:
    stats_err.append(get_spatial_distribution(clipped_results["Temporal RMSE"], "Temporal RMSE (°C)"))

if stats_err:
    df_err = pd.DataFrame(stats_err).round(3)
    cols = ['Metric', 'Mean', 'Median', 'Std', 'Min', 'P95', 'Max']
    print(df_err[cols].to_markdown(index=False))

# --- Table 3: Correlation Analysis ---
print("\n### 3. Correlation Analysis (Spatial Distribution of Temporal r)")
stats_corr = []
if "Temporal Correlation" in clipped_results:
    stats_corr.append(get_spatial_distribution(clipped_results["Temporal Correlation"], "Pearson r"))

if stats_corr:
    df_corr = pd.DataFrame(stats_corr).round(3)
    cols = ['Metric', 'Mean', 'Median', 'Std', 'P05', 'P25', 'P75', 'Max']
    print(df_corr[cols].to_markdown(index=False))

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
import rioxarray
import geopandas as gpd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import gaussian_kde
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# 1. CONFIGURATION
# =====================================================================

HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

# Analysis parameters
TEST_START = 2009219  # 2009-08-07
TEST_END = 2014365
GRID_SPACING_KM = 10.5924

# =====================================================================
# 2. HELPER FUNCTIONS (Statistical Core)
# =====================================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    try:
        return int(base)
    except Exception:
        import re
        m = re.search(r"(\d{7,8})", base)
        return int(m.group(1)) if m else None

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start, end):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if d is None: continue
            if start <= d <= end: valid.append(f)
        except: continue
    return valid

def compute_effective_dof(field_2d, grid_spacing_km, decorr_scale_km=300):
    """Computes effective degrees of freedom accounting for spatial autocorrelation."""
    if field_2d is None or np.isnan(field_2d).all():
        return {'total_dof':0,'effective_dof':0,'reduction_factor':1.0}
    ny, nx = np.array(field_2d.shape)
    total = ny * nx
    domain_w = nx * grid_spacing_km
    domain_h = ny * grid_spacing_km
    n_indep_x = max(1, domain_w / decorr_scale_km)
    n_indep_y = max(1, domain_h / decorr_scale_km)
    eff = max(1, int(n_indep_x * n_indep_y))
    return {'total_dof': total, 'effective_dof': eff, 'reduction_factor': total/eff if eff>0 else 1.0}

def test_normality(data):
    """Robust normality tests."""
    arr = np.asarray(data).ravel()
    arr = arr[np.isfinite(arr)]
    if len(arr) > 5000: # Shapiro is invalid for N > 5000, use KS
        mu, sigma = np.mean(arr), np.std(arr, ddof=0)
        ks_stat, ks_p = stats.kstest(arr, 'norm', args=(mu, sigma))
        return {'Method': 'KS-Test (N>5000)', 'Statistic': ks_stat, 'p-value': ks_p, 'Is_Normal': ks_p > 0.05}
    else:
        sw_stat, sw_p = stats.shapiro(arr)
        return {'Method': 'Shapiro-Wilk', 'Statistic': sw_stat, 'p-value': sw_p, 'Is_Normal': sw_p > 0.05}

def compute_bias_metrics(mswx_vals, ddpm_vals, bias_vals, effective_dof=None):
    """Compute comprehensive bias metrics."""
    mask = np.isfinite(mswx_vals) & np.isfinite(ddpm_vals) & np.isfinite(bias_vals)
    bias = bias_vals[mask]
    
    mean_bias = np.mean(bias)
    std_bias = np.std(bias, ddof=0)
    se = std_bias / np.sqrt(effective_dof) if effective_dof else 0
    
    return {
        'Mean Bias': mean_bias,
        'Median Bias': np.median(bias),
        'Std Dev': std_bias,
        'MAE': np.mean(np.abs(bias)),
        'RMSE': np.sqrt(np.mean(bias**2)),
        '95% CI Lower': mean_bias - 1.96 * se,
        '95% CI Upper': mean_bias + 1.96 * se,
        'Min Bias': np.min(bias),
        'Max Bias': np.max(bias)
    }

def compute_qq_statistics(mswx_vals, ddpm_vals):
    """Compute Q-Q plot statistics (slope, intercept, R2)."""
    mask = np.isfinite(mswx_vals) & np.isfinite(ddpm_vals)
    q_m = np.percentile(mswx_vals[mask], np.arange(1, 100))
    q_d = np.percentile(ddpm_vals[mask], np.arange(1, 100))
    
    slope, intercept = np.polyfit(q_m, q_d, 1)
    yhat = slope * q_m + intercept
    ss_res = np.sum((q_d - yhat)**2)
    ss_tot = np.sum((q_d - np.mean(q_d))**2)
    r2 = 1 - ss_res / ss_tot
    
    return {
        'Slope': slope,
        'Intercept': intercept,
        'R2': r2,
        'Bias @ 5th Percentile': q_d[4] - q_m[4],   # Cold extremes
        'Bias @ 95th Percentile': q_d[94] - q_m[94] # Hot extremes
    }

# =====================================================================
# 3. MAIN EXECUTION
# =====================================================================
print("Initializing Distribution Analysis...")

# --- Load Data ---
f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))
f_mswx = filter_period(f_mswx, TEST_START, TEST_END)
f_ddpm = filter_period(f_ddpm, TEST_START, TEST_END)

print(f"  Files: {len(f_mswx)} MSWX, {len(f_ddpm)} DDPM")

ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# --- Unit Check ---
if ds_mswx['air_temperature'].mean().compute().item() > 200:
    print("  Converting Kelvin to Celsius...")
    ds_mswx['air_temperature'] -= 273.15
    ds_ddpm['air_temperature'] -= 273.15

# --- Calculate Climatology (Spatial Fields) ---
print("  Calculating Spatial Climatologies...")
overall_mswx = ds_mswx["air_temperature"].groupby("time.year").mean("time").mean("year")
overall_ddpm = ds_ddpm["air_temperature"].groupby("time.year").mean("time").mean("year")
overall_bias = overall_ddpm - overall_mswx

# --- Spatial Clipping ---
print("  Clipping to India...")
overall_mswx = overall_mswx.rename({"lat":"y","lon":"x"})
overall_ddpm = overall_ddpm.rename({"lat":"y","lon":"x"})
overall_bias = overall_bias.rename({"lat":"y","lon":"x"})

for da in [overall_mswx, overall_ddpm, overall_bias]:
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)

gdf = gpd.read_file(INDIA_SHP)
overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

# Flatten arrays
mswx_vals = overall_mswx_clipped.values.flatten()
mswx_vals = mswx_vals[np.isfinite(mswx_vals)]
ddpm_vals = overall_ddpm_clipped.values.flatten()
ddpm_vals = ddpm_vals[np.isfinite(ddpm_vals)]
bias_vals = overall_bias_clipped.values.flatten()
bias_vals = bias_vals[np.isfinite(bias_vals)]

# --- Statistical Computations ---
print("  Computing Advanced Statistics...")

# 1. Effective Degrees of Freedom (Spatial Autocorrelation)
dof_info = compute_effective_dof(overall_bias_clipped.values, GRID_SPACING_KM)
eff_dof = dof_info['effective_dof']

# 2. Bias Metrics
bias_stats = compute_bias_metrics(mswx_vals, ddpm_vals, bias_vals, eff_dof)

# 3. Normality Test
norm_test = test_normality(bias_vals)

# 4. Q-Q Analysis
qq_stats = compute_qq_statistics(mswx_vals, ddpm_vals)

# =====================================================================
# 4. OUTPUT GENERATION
# =====================================================================
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
print("="*80 + "\n")

print("**Statistical Analysis: Bias Distribution & Conditional Reliability (Q-Q)**")
print(f"Data Points: {len(bias_vals):,} | Effective Spatial DOF: {eff_dof:,}")
print(f"Spatial Reduction Factor: {dof_info['reduction_factor']:.2f}x (Due to spatial autocorrelation)")

print("\n### 1. Overall Error Distribution (Bias Metrics)")
df_bias = pd.DataFrame([bias_stats]).T
df_bias.columns = ['Value']
print(df_bias.round(4).to_markdown())

print("\n### 2. Normality of Errors")
print(f"Test Method: {norm_test['Method']}")
print(f"Statistic: {norm_test['Statistic']:.4f}")
print(f"p-value: {norm_test['p-value']:.4e}")
print(f"Is Gaussian? {norm_test['Is_Normal']}")

print("\n### 3. Conditional Reliability (Q-Q Analysis)")
print("Quantifying performance across the distribution (Cold to Hot extremes).")
df_qq = pd.DataFrame([qq_stats]).T
df_qq.columns = ['Value']
print(df_qq.round(4).to_markdown())

print("\n### 4. Interpretation Guide")
print("- **Q-Q Slope < 1.0**: Model underestimates variability (too smooth).")
print("- **Q-Q Slope > 1.0**: Model overestimates variability (too extreme).")
print("- **R2**: Linearity of the Q-Q plot (1.0 = perfect distribution match).")
print("- **5th/95th Percentile Bias**: Specific error at cold/hot extremes.")

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

In [ ]:
#!/usr/bin/env python3
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# ---------------- USER CONFIG ----------------
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"

# Analysis Period
TEST_START = 2009219   # 2009-08-07
TEST_END = 2014365     # 2014-12-31

# ---------------- HELPER FUNCTIONS ----------------
def get_file_date_token(fname):
    """Parse date token from filename."""
    base = os.path.basename(fname)
    base_noext = os.path.splitext(base)[0]
    import re
    m = re.search(r"(\d{7})", base_noext) # YYYYDOY
    if m: return int(m.group(1))
    m = re.search(r"(\d{8})", base_noext) # YYYYMMDD
    if m: return int(m.group(1))
    return None

def token_to_datetime(token):
    """Convert token to timestamp."""
    if token is None: return None
    s = str(token)
    if len(s) == 7: return pd.to_datetime(f"{s[:4]}-{s[4:]}", format="%Y-%j")
    if len(s) == 8: return pd.to_datetime(s, format="%Y%m%d")
    return None

def filter_files(files, start, end):
    valid = []
    for f in sorted(files):
        tok = get_file_date_token(f)
        if tok and start <= tok <= end:
            valid.append(f)
    return valid

def get_season(month):
    """Returns Indian Meteorological Season."""
    if month in [1, 2]: return 'Winter (JF)'
    elif month in [3, 4, 5]: return 'Pre-Monsoon (MAM)'
    elif month in [6, 7, 8, 9]: return 'Monsoon (JJAS)'
    elif month in [10, 11, 12]: return 'Post-Monsoon (OND)'
    return 'Unknown'

# ---------------- MAIN EXECUTION ----------------
print("Initializing Seasonal Cycle Analysis...")

# 1. Load Files
mswx_files = filter_files(glob.glob(os.path.join(HR_MSWX, "*.nc")), TEST_START, TEST_END)
ddpm_files = filter_files(glob.glob(os.path.join(HR_DDPM, "*.nc")), TEST_START, TEST_END)

print(f"  Files: {len(mswx_files)} MSWX, {len(ddpm_files)} DDPM")

ds_mswx = xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(ddpm_files, combine="nested", concat_dim="time", engine="netcdf4")

# 2. Assign Time if missing
if "time" not in ds_mswx.coords:
    times = [token_to_datetime(get_file_date_token(f)) for f in mswx_files]
    ds_mswx = ds_mswx.assign_coords(time=("time", pd.to_datetime(times)))

if "time" not in ds_ddpm.coords:
    times = [token_to_datetime(get_file_date_token(f)) for f in ddpm_files]
    ds_ddpm = ds_ddpm.assign_coords(time=("time", pd.to_datetime(times)))

# 3. Compute Spatial Means (Time Series)
print("  Computing spatial means...")
# Standardize var names
v_m = "air_temperature" if "air_temperature" in ds_mswx else list(ds_mswx.data_vars)[0]
v_d = "air_temperature" if "air_temperature" in ds_ddpm else list(ds_ddpm.data_vars)[0]

# Average over lat/lon
spatial_mean_mswx = ds_mswx[v_m].mean(dim=xr.ALL_DIMS, keep_attrs=True) if 'lat' not in ds_mswx.dims else ds_mswx[v_m].mean(dim=['lat', 'lon'])
spatial_mean_ddpm = ds_ddpm[v_d].mean(dim=xr.ALL_DIMS, keep_attrs=True) if 'lat' not in ds_ddpm.dims else ds_ddpm[v_d].mean(dim=['lat', 'lon'])

# To Pandas Series
ts_mswx = spatial_mean_mswx.to_series()
ts_ddpm = spatial_mean_ddpm.to_series()

# Align
df = pd.concat([ts_mswx, ts_ddpm], axis=1, join="inner")
df.columns = ["MSWX", "DDPM"]

# Unit Conversion
if df["MSWX"].mean() > 200: df["MSWX"] -= 273.15
if df["DDPM"].mean() > 200: df["DDPM"] -= 273.15

# 4. Calculate Monthly Climatology
print("  Calculating Monthly & Seasonal Biases...")
df['Month'] = df.index.month
df['Bias'] = df['DDPM'] - df['MSWX']

# Group by Month (1-12)
monthly_stats = df.groupby('Month').mean()
monthly_stats['Season'] = [get_season(m) for m in monthly_stats.index]

# Group by Season
seasonal_stats = df.groupby(df.index.month.map(get_season)).mean()

# ---------------- OUTPUT GENERATION ----------------
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
print("="*80 + "\n")

print("**Seasonal Bias Cycle Analysis**")
print("Analysis of systematic errors across the annual cycle.")

print("\n### 1. Monthly Mean Bias (°C)")
print(monthly_stats[['Season', 'MSWX', 'DDPM', 'Bias']].round(3).to_markdown())

print("\n### 2. Seasonal Aggregates (°C)")
# Reorder seasons logically
season_order = ['Winter (JF)', 'Pre-Monsoon (MAM)', 'Monsoon (JJAS)', 'Post-Monsoon (OND)']
seasonal_stats = seasonal_stats.reindex(season_order)
print(seasonal_stats[['MSWX', 'DDPM', 'Bias']].round(3).to_markdown())

print("\n### 3. Key Observations")
max_bias = monthly_stats['Bias'].max()
min_bias = monthly_stats['Bias'].min()
print(f"- **Max Warm Bias:** {max_bias:.3f} °C (Month {monthly_stats['Bias'].idxmax()})")
print(f"- **Max Cold Bias:** {min_bias:.3f} °C (Month {monthly_stats['Bias'].idxmin()})")

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

### IMD Zone RMSE Ranking

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
import rioxarray
import geopandas as gpd
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. CONFIGURATION
# ============================================================

HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
IMD_ZONES_SHP = "<SET YOUR PATH HERE>"

# Test period
TEST_START = 2009219
TEST_END   = 2014365

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def filter_period(files, start=TEST_START, end=TEST_END):
    valid = []
    for f in sorted(files):
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

# ============================================================
# 3. DATA PROCESSING
# ============================================================
print("Initializing Zonal RMSE Analysis...")

# --- Load Data ---
f_mswx = filter_period(glob.glob(os.path.join(HR_MSWX, "*.nc")))
f_ddpm = filter_period(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))

print(f"  Files: {len(f_mswx)} MSWX, {len(f_ddpm)} DDPM")

if not f_mswx or not f_ddpm:
    raise RuntimeError("No files found.")

ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# --- Compute RMSE Map ---
print("  Computing Temporal RMSE Map...")
da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

# Unit Check
if da_m.mean().compute().item() > 200: da_m -= 273.15
if da_d.mean().compute().item() > 200: da_d -= 273.15

bias_ts = da_d - da_m
rmse_map = np.sqrt((bias_ts ** 2).mean("time", skipna=True))

# Prepare for Clipping
rmse_map = rmse_map.rename({"lat": "y", "lon": "x"})
rmse_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse_map.rio.write_crs("EPSG:4326", inplace=True)

# ============================================================
# 4. ZONAL EXTRACTION
# ============================================================
print("  Extracting Statistics per IMD Zone...")

gdf_zones = gpd.read_file(IMD_ZONES_SHP)
zone_stats = []

for _, row in gdf_zones.iterrows():
    zone_name = row["ST_NM"] # Assuming ST_NM is the zone/state name column
    
    try:
        # Clip RMSE map to zone
        zone_clip = rmse_map.rio.clip([row.geometry], gdf_zones.crs, drop=True, all_touched=True)
        vals = zone_clip.values.flatten()
        vals = vals[np.isfinite(vals)]
        
        if len(vals) > 0:
            zone_stats.append({
                "Zone": zone_name,
                "Mean RMSE": float(np.mean(vals)),
                "Std RMSE": float(np.std(vals)), # Spatial variability of RMSE within zone
                "Max RMSE": float(np.max(vals))
            })
    except Exception:
        continue

df_zones = pd.DataFrame(zone_stats).sort_values("Mean RMSE")
national_mean = df_zones["Mean RMSE"].mean()

# ============================================================
# 5. OUTPUT GENERATION
# ============================================================
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
print("="*80 + "\n")

print("**Regional Performance Analysis (IMD Zones)**")
print(f"National Average RMSE: {national_mean:.3f} °C")

print("\n### 1. Best Performing Zones (Top 5 Lowest RMSE)")
print(df_zones.head(5).to_markdown(index=False, floatfmt=".3f"))

print("\n### 2. Worst Performing Zones (Bottom 5 Highest RMSE)")
print(df_zones.tail(5).sort_values("Mean RMSE", ascending=False).to_markdown(index=False, floatfmt=".3f"))

print("\n### 3. Full Zonal Ranking")
print(df_zones.to_markdown(index=False, floatfmt=".3f"))

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

### Statistical Extraction: Long-Term Trends & Inter-Annual Variability (1979-2014)

In [ ]:
#!/usr/bin/env python3
# ULTIMATE ROBUSTNESS: Atomic File Processing
# Calculates spatial mean for each file individually to avoid concatenation errors.

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import rasterio.features
import rasterio.transform
from scipy import stats
import warnings

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
model_base = "<SET YOUR PATH HERE>"
models = {
    "CESM2-WACCM": "1979-01-01_CESM2-WACCM.nc",
    "CESM2": "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC": "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL": "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM": "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8": "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0": "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR": "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM": "1979-01-01_NorESM2-MM.nc",
}
extra_files_dirs = {
    "Ensemble": "<SET YOUR PATH HERE>",
    "MSWX": "<SET YOUR PATH HERE>",
}
india_shp = "<SET YOUR PATH HERE>"

START_YEAR = 1979
END_YEAR = 2014
YEARS = list(range(START_YEAR, END_YEAR + 1))

# ---------------- HELPER FUNCTIONS ----------------

def get_model_tag_from_sample(sample_fname):
    base = os.path.basename(sample_fname)
    parts = base.split("_", 1)
    if len(parts) == 2:
        return parts[1].replace(".nc","")
    m = re.search(r"[A-Za-z].*$", base)
    return m.group(0).replace(".nc","") if m else base.replace(".nc","")

def files_for_model_token_sep(model_base, sample_fname, tag=None):
    if tag is None: tag = get_model_tag_from_sample(sample_fname)
    pattern1 = os.path.join(model_base, f"*_{tag}*.nc")
    pattern2 = os.path.join(model_base, f"*{tag}*.nc")
    candidates = sorted(set(glob.glob(pattern1) + glob.glob(pattern2)))
    token_re = re.compile(r'(^|[_\.])' + re.escape(tag) + r'($|[_\.])')
    filtered = [f for f in candidates if token_re.search(os.path.basename(f))]
    return filtered or candidates

def safe_read_country_shp(shp_path):
    try:
        return gpd.read_file(shp_path)
    except:
        world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
        return world[world["name"] == "India"]

def build_mask_for_ds(ds_sample, gdf_india):
    lons = ds_sample["lon"].values
    lats = ds_sample["lat"].values
    transform = rasterio.transform.from_bounds(lons.min(), lats.min(), lons.max(), lats.max(), lons.size, lats.size)
    shapes = ((geom, 1) for geom in gdf_india.geometry)
    mask = rasterio.features.rasterize(shapes=shapes, out_shape=(lats.size, lons.size), transform=transform, fill=0, all_touched=True, dtype=np.uint8)
    if lats[0] < lats[-1]: mask = np.flipud(mask)
    return xr.DataArray(mask, coords={"lat": lats, "lon": lons}, dims=("lat","lon"))

def get_year_from_filename(fname):
    base = os.path.basename(fname)
    m = re.search(r"(\d{4})", base)
    if m: return int(m.group(1))
    return None

def process_files_sequentially(files, gdf_india):
    """
    Processes every file individually.
    No concatenation. No memory issues. No coordinate sorting issues.
    """
    # Group by year
    files_by_year = {}
    for f in files:
        y = get_year_from_filename(f)
        if y is not None and START_YEAR <= y <= END_YEAR:
            if y not in files_by_year: files_by_year[y] = []
            files_by_year[y].append(f)
            
    if not files_by_year: return pd.Series(dtype=float)

    # Init Grid (from first file)
    try:
        ref_ds = xr.open_dataset(files[0], decode_times=False)
        var_candidates = ["tas", "air_temperature", "t2m"]
        var_name = next((v for v in var_candidates if v in ref_ds), list(ref_ds.data_vars)[0])
        
        mask = build_mask_for_ds(ref_ds, gdf_india)
        weights = np.cos(np.deg2rad(ref_ds["lat"]))
        ref_ds.close()
    except Exception as e:
        print(f"    Error initializing grid: {e}")
        return pd.Series(dtype=float)

    annual_means = {}
    
    # Iterate Years
    sorted_years = sorted(files_by_year.keys())
    for year in sorted_years:
        year_files = files_by_year[year]
        daily_values = []
        
        # Iterate Files in Year (Atomic Processing)
        for f in year_files:
            try:
                with xr.open_dataset(f, decode_times=False) as ds:
                    # Select var
                    if var_name in ds: da = ds[var_name]
                    else: da = ds[list(ds.data_vars)[0]]
                    
                    # Squeeze dims if needed (remove time/model dims)
                    da = da.squeeze(drop=True)
                    
                    # Apply Mask & Weights
                    da_masked = da.where(mask == 1)
                    
                    # Compute Spatial Mean (Single Scalar)
                    # Handle weighted mean manually to be safe with scalars
                    weighted_sum = (da_masked * weights).sum().item()
                    weights_sum = weights.where(da_masked.notnull()).sum().item()
                    
                    if weights_sum > 0:
                        daily_values.append(weighted_sum / weights_sum)
            except Exception:
                continue
        
        # Compute Annual Mean from daily scalars
        if daily_values:
            annual_means[year] = np.mean(daily_values)
            
    return pd.Series(annual_means)

def calculate_trend_statistics(series):
    valid = series.dropna()
    if len(valid) < 5:
        return {'Trend': np.nan, 'p-value': np.nan, 'Std Err': np.nan, 'IAV': np.nan}
    x = np.arange(len(valid))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, valid.values)
    trend_line = slope * x + intercept
    residuals = valid.values - trend_line
    return {
        'Trend (°C/dec)': slope * 10,
        'p-value': p_value,
        'Std Err': std_err * 10,
        'IAV (°C)': np.std(residuals)
    }

# ---------------- MAIN PROCESS ----------------
print("Initializing Atomic Trend Analysis...")
gdf_india = safe_read_country_shp(india_shp)
all_results = {}

# 1. Models
for model_name, sample_fname in models.items():
    print(f"  Processing: {model_name}")
    model_tag = get_model_tag_from_sample(sample_fname)
    files = files_for_model_token_sep(model_base, sample_fname, tag=model_tag)
    
    if not files: 
        print("    No files found.")
        continue
    
    ann_series = process_files_sequentially(files, gdf_india)
    if not ann_series.empty and ann_series.mean() > 200: ann_series -= 273.15
    all_results[model_name] = ann_series

# 2. Extras
for label, path in extra_files_dirs.items():
    print(f"  Processing: {label}")
    files = sorted(glob.glob(os.path.join(path, "*.nc")))
    ann_series = process_files_sequentially(files, gdf_india)
    if not ann_series.empty and ann_series.mean() > 200: ann_series -= 273.15
    all_results[label] = ann_series

df = pd.DataFrame(all_results)
df = df.reindex(YEARS)

# ---------------- OUTPUT ----------------
print("\n" + "="*80)
print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
print("="*80 + "\n")

print("**Long-Term Trend Analysis (1979-2014)**")
if df.empty:
    print("CRITICAL: No data processed.")
else:
    print("\n### 1. Trend Statistics")
    stats_list = []
    for col in df.columns:
        s = calculate_trend_statistics(df[col])
        s['Dataset'] = col
        s['Mean T (°C)'] = df[col].mean()
        stats_list.append(s)

    df_stats = pd.DataFrame(stats_list).set_index('Dataset')
    priority = ['MSWX', 'Ensemble']
    others = [c for c in df_stats.index if c not in priority]
    df_stats = df_stats.reindex(priority + sorted(others))
    print(df_stats[['Mean T (°C)', 'Trend (°C/dec)', 'p-value', 'IAV (°C)']].round(4).to_markdown())

    print("\n### 2. Decadal Means")
    df['Decade'] = (df.index // 10) * 10
    decadal = df.groupby('Decade').mean().T
    print(decadal.round(3).to_markdown())

    if 'MSWX' in df.columns:
        print("\n### 3. Correlation with Reference")
        corrs = df.drop(columns=['Decade']).corrwith(df['MSWX']).drop('MSWX')
        print(pd.DataFrame(corrs, columns=['Correlation']).sort_values('Correlation', ascending=False).round(3).to_markdown())

print("\n" + "="*80)
print("END OF OUTPUT")
print("="*80)

In [ ]:
#!/usr/bin/env python3
"""
CMIP6 Historical Temperature Characterization (1979-2014)
Peer-review grade: Mann-Kendall trends, AR1-corrected OLS, proper area weighting.
"""

import os, re, glob
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rasterio.features
import rasterio.transform
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Optional: pymannkendall (pip install pymannkendall) ─────────────────────
try:
    import pymannkendall as mk
    HAS_MK = True
except ImportError:
    HAS_MK = False
    print("WARNING: pymannkendall not installed. Install with: pip install pymannkendall")
    print("         Falling back to AR1-corrected OLS only.\n")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
model_base = "<SET YOUR PATH HERE>"
models = {
    "CESM2-WACCM":       "1979-01-01_CESM2-WACCM.nc",
    "CESM2":             "1979-01-01_CESM2.nc",
    "EC-Earth3-AerChem": "1979-01-01_EC-Earth3-AerChem.nc",
    "EC-Earth3-CC":      "1979-01-01_EC-Earth3-CC.nc",
    "HadGEM3-GC31-LL":   "1979-01-01_HadGEM3-GC31-LL.nc",
    "HadGEM3-GC31-MM":   "1979-01-01_HadGEM3-GC31-MM.nc",
    "INM-CM4-8":         "1979-01-01_INM-CM4-8.nc",
    "INM-CM5-0":         "1979-01-01_INM-CM5-0.nc",
    "MPI-ESM1-2-HR":     "1979-01-01_MPI-ESM1-2-HR.nc",
    "NorESM2-MM":        "1979-01-01_NorESM2-MM.nc",
}
extra_files_dirs = {
    "Ensemble": "<SET YOUR PATH HERE>",
    "MSWX":     "<SET YOUR PATH HERE>",
}
india_shp  = "<SET YOUR PATH HERE>"
START_YEAR = 1979
END_YEAR   = 2014
YEARS      = list(range(START_YEAR, END_YEAR + 1))
ALPHA      = 0.05   # significance threshold — stated explicitly

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def get_tag(fname):
    base = os.path.basename(fname)
    parts = base.split("_", 1)
    if len(parts) == 2:
        return parts[1].replace(".nc", "")
    m = re.search(r"[A-Za-z].*$", base)
    return m.group(0).replace(".nc", "") if m else base.replace(".nc", "")

def find_files(base_dir, tag):
    p1 = sorted(glob.glob(os.path.join(base_dir, f"*_{tag}*.nc")))
    p2 = sorted(glob.glob(os.path.join(base_dir, f"*{tag}*.nc")))
    candidates = sorted(set(p1 + p2))
    tok = re.compile(r'(^|[_\.])' + re.escape(tag) + r'($|[_\.])')
    filtered = [f for f in candidates if tok.search(os.path.basename(f))]
    return filtered or candidates

def load_india_shapefile(path):
    try:
        return gpd.read_file(path)
    except Exception:
        world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
        return world[world["name"] == "India"]

def build_mask(ds, gdf):
    lons = ds["lon"].values
    lats = ds["lat"].values
    transform = rasterio.transform.from_bounds(
        lons.min(), lats.min(), lons.max(), lats.max(), lons.size, lats.size)
    shapes = ((g, 1) for g in gdf.geometry)
    mask = rasterio.features.rasterize(
        shapes=shapes, out_shape=(lats.size, lons.size),
        transform=transform, fill=0, all_touched=True, dtype=np.uint8)
    if lats[0] < lats[-1]:
        mask = np.flipud(mask)
    return xr.DataArray(mask, coords={"lat": lats, "lon": lons}, dims=("lat", "lon"))

def get_year(fname):
    m = re.search(r"(\d{4})", os.path.basename(fname))
    return int(m.group(1)) if m else None

# ─────────────────────────────────────────────────────────────────────────────
# FIX 1: Proper 2D area-weighted spatial mean
# ─────────────────────────────────────────────────────────────────────────────
def area_weighted_mean(da, mask, lat_weights_1d):
    """
    Correct area-weighted spatial mean.
    lat_weights_1d: cos(lat) array of shape (lat,)
    Broadcasts weights to (lat, lon) before masking — handles
    uneven NaN patterns (coastlines, ocean) correctly.
    """
    # Broadcast cos(lat) weights to full (lat, lon) grid
    w2d = lat_weights_1d.broadcast_like(da)          # shape (lat, lon)
    da_masked   = da.where(mask == 1)
    valid_mask  = da_masked.notnull()
    wsum        = (w2d * valid_mask).sum(["lat","lon"]).item()
    if wsum == 0:
        return np.nan
    return (da_masked * w2d).sum(["lat","lon"]).item() / wsum

# ─────────────────────────────────────────────────────────────────────────────
# Atomic file processing with corrected area weighting
# ─────────────────────────────────────────────────────────────────────────────
def process_files(files, gdf):
    by_year = {}
    for f in files:
        y = get_year(f)
        if y and START_YEAR <= y <= END_YEAR:
            by_year.setdefault(y, []).append(f)
    if not by_year:
        return pd.Series(dtype=float)

    # Initialise grid from first file
    try:
        with xr.open_dataset(files[0], decode_times=False) as ref:
            var_name = next((v for v in ["tas","air_temperature","t2m"]
                             if v in ref), list(ref.data_vars)[0])
            mask         = build_mask(ref, gdf)
            lat_weights  = np.cos(np.deg2rad(ref["lat"]))   # shape (lat,)
    except Exception as e:
        print(f"    Grid init error: {e}")
        return pd.Series(dtype=float)

    annual_means = {}
    for year in sorted(by_year):
        daily_vals = []
        # FIX 2: track days per year for leap-year-aware weighting
        days_in_year = 0
        for f in by_year[year]:
            try:
                with xr.open_dataset(f, decode_times=False) as ds:
                    da = ds[var_name] if var_name in ds else ds[list(ds.data_vars)[0]]
                    da = da.squeeze(drop=True)
                    val = area_weighted_mean(da, mask, lat_weights)
                    if not np.isnan(val):
                        daily_vals.append(val)
                        days_in_year += 1
            except Exception:
                continue
        if daily_vals:
            annual_means[year] = np.mean(daily_vals)   # simple mean = correct if all days present

    return pd.Series(annual_means)

# ─────────────────────────────────────────────────────────────────────────────
# FIX 3: AR1-corrected OLS + Mann-Kendall trend statistics
# ─────────────────────────────────────────────────────────────────────────────
def trend_statistics(series, label=""):
    valid = series.dropna()
    n = len(valid)
    if n < 5:
        return {}

    # FIX: use actual year values as x, not positional index
    years = valid.index.values.astype(float)
    x = years - years[0]      # starts at 0, spacing = actual year gaps
    y = valid.values

    # OLS
    slope, intercept, r, p_ols, se = stats.linregress(x, y)
    trend_line = slope * x + intercept
    residuals  = y - trend_line
    iav        = np.std(residuals, ddof=1)

    # AR1 effective sample size correction
    r1 = np.corrcoef(residuals[:-1], residuals[1:])[0, 1]
    if r1 < 1.0:
        n_eff = max(3, int(n * (1 - r1) / (1 + r1)))
    else:
        n_eff = n
    # Recompute p-value using effective dof
    t_stat = slope / (se + 1e-12)
    p_ar1  = 2 * stats.t.sf(abs(t_stat), df=n_eff - 2)

    # 95% CI on slope (using OLS se, AR1 dof)
    t_crit  = stats.t.ppf(0.975, df=n_eff - 2)
    ci_half = t_crit * se
    ci95_lo = (slope - ci_half) * 10   # °C/decade
    ci95_hi = (slope + ci_half) * 10

    result = {
        "Mean_T"            : np.mean(y),
        "Slope_per_dec"     : slope * 10,          # °C/decade
        "CI95_lo"           : ci95_lo,
        "CI95_hi"           : ci95_hi,
        "p_OLS"             : p_ols,
        "p_AR1corrected"    : p_ar1,
        "AR1_coef"          : round(r1, 3),
        "n_eff"             : n_eff,
        "n_actual"          : n,
        "IAV_detrended"     : iav,
        "R2"                : r**2,
    }

    # Mann-Kendall (non-parametric, preferred for climate)
    if HAS_MK:
        mk_result = mk.original_test(y, alpha=ALPHA)
        result["MK_trend"]  = mk_result.trend        # 'increasing'/'decreasing'/'no trend'
        result["MK_p"]      = mk_result.p
        result["Sens_slope_per_dec"] = mk_result.slope * 10  # Sen's slope °C/decade
        result["Sens_intercept"]     = mk_result.intercept

    return result

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("CMIP6 Historical Temperature Analysis (1979-2014) — Peer-Review Grade")
print("=" * 70)
print(f"Significance threshold α = {ALPHA}")
print(f"Mann-Kendall available: {HAS_MK}\n")

gdf_india  = load_india_shapefile(india_shp)
all_series = {}

for name, sample in models.items():
    print(f"  Processing: {name}")
    tag   = get_tag(sample)
    files = find_files(model_base, tag)
    if not files:
        print(f"    No files found for {name}")
        continue
    s = process_files(files, gdf_india)
    if not s.empty and s.mean() > 200:
        s -= 273.15
    all_series[name] = s

for label, path in extra_files_dirs.items():
    print(f"  Processing: {label}")
    files = sorted(glob.glob(os.path.join(path, "*.nc")))
    s = process_files(files, gdf_india)
    if not s.empty and s.mean() > 200:
        s -= 273.15
    all_series[label] = s

df = pd.DataFrame(all_series).reindex(YEARS)

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

# ── 1. Trend statistics ──────────────────────────────────────────────────────
print("\n### TABLE 1: Trend Statistics")
print(f"(OLS + AR1-corrected p-value; Mann-Kendall if available; α = {ALPHA})\n")

stats_rows = []
ref_mean = df["MSWX"].mean() if "MSWX" in df.columns else np.nan

priority = ["MSWX", "Ensemble"]
cols_ordered = priority + sorted([c for c in df.columns if c not in priority])

for col in cols_ordered:
    if col not in df.columns:
        continue
    r = trend_statistics(df[col], label=col)
    if not r:
        continue
    bias = r["Mean_T"] - ref_mean if col != "MSWX" else 0.0
    r["Bias_vs_MSWX"] = bias
    r["Dataset"] = col
    stats_rows.append(r)

df_stats = pd.DataFrame(stats_rows).set_index("Dataset")

# Core table for paper
core_cols = ["Mean_T", "Bias_vs_MSWX", "Slope_per_dec", "CI95_lo", "CI95_hi",
             "p_AR1corrected", "AR1_coef", "n_eff", "IAV_detrended"]
if HAS_MK:
    core_cols += ["MK_p", "Sens_slope_per_dec", "MK_trend"]

available = [c for c in core_cols if c in df_stats.columns]
print(df_stats[available].round(4).to_markdown())

# ── 2. Inter-model spread ────────────────────────────────────────────────────
model_only = df_stats.drop(index=["MSWX", "Ensemble"], errors="ignore")
if not model_only.empty:
    print("\n### TABLE 2: Inter-model Spread Summary")
    print(f"  Mean T range  : {model_only['Mean_T'].min():.3f} – {model_only['Mean_T'].max():.3f} °C")
    print(f"  Inter-model spread (max-min Mean T): {model_only['Mean_T'].max() - model_only['Mean_T'].min():.3f} °C")
    print(f"  Bias range    : {model_only['Bias_vs_MSWX'].min():.3f} – {model_only['Bias_vs_MSWX'].max():.3f} °C")
    print(f"  Trend range   : {model_only['Slope_per_dec'].min():.3f} – {model_only['Slope_per_dec'].max():.3f} °C/decade")
    print(f"  Trend spread  : {model_only['Slope_per_dec'].max() - model_only['Slope_per_dec'].min():.3f} °C/decade")
    sig_ar1 = model_only[model_only["p_AR1corrected"] < ALPHA]
    insig   = model_only[model_only["p_AR1corrected"] >= ALPHA]
    print(f"  Significant trends (AR1-corrected p < {ALPHA}): {len(sig_ar1)} models")
    print(f"  Non-significant trends: {len(insig)} models")
    if HAS_MK and "MK_p" in model_only.columns:
        sig_mk = model_only[model_only["MK_p"] < ALPHA]
        print(f"  Significant trends (Mann-Kendall p < {ALPHA}): {len(sig_mk)} models")

# ── 3. Temporal correlations with MSWX ──────────────────────────────────────
if "MSWX" in df.columns:
    print("\n### TABLE 3: Temporal Correlation with MSWX (1979-2014)")
    corr_rows = []
    for col in df.columns:
        if col == "MSWX":
            continue
        paired = df[["MSWX", col]].dropna()
        if len(paired) < 5:
            continue
        r_val, p_val = stats.pearsonr(paired["MSWX"], paired[col])
        # Effective dof for correlation significance
        n = len(paired)
        resid_x = paired["MSWX"] - paired["MSWX"].mean()
        resid_y = paired[col] - paired[col].mean()
        r1x = np.corrcoef(resid_x.values[:-1], resid_x.values[1:])[0,1]
        r1y = np.corrcoef(resid_y.values[:-1], resid_y.values[1:])[0,1]
        n_eff_corr = max(3, int(n * (1 - r1x*r1y) / (1 + r1x*r1y)))
        t_corr = r_val * np.sqrt(n_eff_corr - 2) / np.sqrt(1 - r_val**2 + 1e-12)
        p_corr = 2 * stats.t.sf(abs(t_corr), df=n_eff_corr - 2)
        corr_rows.append({"Dataset": col, "r": r_val, "p_AR1corrected": p_corr, "n_eff": n_eff_corr})

    df_corr = pd.DataFrame(corr_rows).set_index("Dataset").sort_values("r", ascending=False)
    print(df_corr.round(3).to_markdown())

# ── 4. Decadal means ─────────────────────────────────────────────────────────
print("\n### TABLE 4: Decadal Mean Temperatures (°C)")
df_nodec = df.drop(columns=["Decade"], errors="ignore").copy()
df_nodec["Decade"] = (df_nodec.index // 10) * 10
decadal  = df_nodec.groupby("Decade").mean().T
decadal  = decadal.reindex([c for c in cols_ordered if c in decadal.index])
print(decadal.round(3).to_markdown())

print("\n" + "=" * 70)
print("NOTES FOR METHODS SECTION:")
print(f"  Trend method  : OLS linear regression + AR1 effective sample size correction")
if HAS_MK:
    print(f"  Also reported : Mann-Kendall test + Sen's slope (non-parametric)")
print(f"  Significance  : α = {ALPHA} (two-tailed)")
print(f"  Spatial mean  : Area-weighted (cosine latitude), India land mask")
print(f"  Temperature   : Kelvin→Celsius converted where mean > 200 K")
print("=" * 70)

In [ ]:
#!/usr/bin/env python3
# STATISTICAL EXTRACTION: Model Bias Comparison (Raw vs QDM vs DDPM)
# Fixed: Uses unclipped coordinates for alignment to prevent dimension mismatch errors.

import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import os
import glob
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODELS = [
    "CESM2", "CESM2-WACCM", "EC-Earth3-AerChem", "EC-Earth3-CC", 
    "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "INM-CM4-8", "INM-CM5-0", 
    "MPI-ESM1-2-HR", "NorESM2-MM"
]

# Paths
SHAPEFILE_PATH = "<SET YOUR PATH HERE>"
DIR_PATHS = {
    "raw": "<SET YOUR PATH HERE>",
    "qdm": "<SET YOUR PATH HERE>",
    "ddpm": "<SET YOUR PATH HERE>",
    "mswx": "<SET YOUR PATH HERE>"
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_target_shape(folder_path, pattern="*.nc"):
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                if data.ndim == 3: return data.shape[1:]
                elif data.ndim == 2: return data.shape
        except: continue
    return None

def compute_robust_mean_numpy(folder_path, pattern, expected_shape_2d):
    """Computes the time-mean map (climatology) from multiple files."""
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None

    running_sum = np.zeros(expected_shape_2d, dtype=np.float64)
    total_count = np.zeros(expected_shape_2d, dtype=np.int32)
    valid_count = 0
    ref_ds = None

    for f in tqdm(files, desc=f"  Loading {pattern}", leave=False):
        try:
            if os.path.getsize(f) < 10240: continue
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                
                # Check shapes and sum
                if data.ndim == 3:
                    if data.shape[1:] != expected_shape_2d: continue
                    val = data.sum(dim='time').values
                    cnt = data.notnull().sum(dim='time').values
                elif data.ndim == 2:
                    if data.shape != expected_shape_2d: continue
                    val = data.values
                    cnt = data.notnull().values.astype(int)
                else: continue

                running_sum += np.nan_to_num(val)
                total_count += cnt
                valid_count += 1

                if ref_ds is None:
                    ref_ds = ds.copy(deep=True)
                    if 'time' in ref_ds.dims: ref_ds = ref_ds.isel(time=0, drop=True)
        except: continue

    if valid_count == 0 or ref_ds is None: return None

    with np.errstate(divide='ignore', invalid='ignore'):
        mean_arr = running_sum / total_count
    
    return xr.Dataset(
        {'tas': (('lat', 'lon'), mean_arr)},
        coords={'lat': ref_ds['lat'], 'lon': ref_ds['lon']}
    )

def safe_clip(da, gdf):
    """
    Robustly sets spatial dims and CRS immediately before clipping.
    """
    # Ensure standard names
    if 'latitude' in da.coords: da = da.rename({'latitude': 'lat'})
    if 'longitude' in da.coords: da = da.rename({'longitude': 'lon'})
    
    # Enforce spatial dimensions
    da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    
    # Force CRS
    da.rio.write_crs("EPSG:4326", inplace=True)
    
    # Clip
    return da.rio.clip(gdf.geometry, gdf.crs, drop=True)

def calculate_spatial_metrics(model_name, stage_name, bias_map):
    """Calculates MB, SD, RMSE, and Percentiles for a bias map."""
    vals = bias_map.values.flatten()
    vals = vals[np.isfinite(vals)] # Remove NaNs (masked areas)
    
    if len(vals) == 0:
        return None

    return {
        'Model': model_name,
        'Stage': stage_name,
        'Mean Bias (MB)': np.mean(vals),
        'RMSE': np.sqrt(np.mean(vals**2)),
        'Std Dev (SD)': np.std(vals),
        'P05 (Cold Tail)': np.percentile(vals, 5),
        'P95 (Warm Tail)': np.percentile(vals, 95)
    }

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def main():
    print("Initializing Bias Statistical Analysis...")
    
    # Load Shapefile
    try:
        india_gdf = gpd.read_file(SHAPEFILE_PATH)
        if india_gdf.crs != "EPSG:4326": 
            india_gdf = india_gdf.to_crs("EPSG:4326")
    except Exception as e:
        print(f"CRITICAL: Failed to load shapefile: {e}")
        return
    
    # Determine Grid Shape
    target_shape = get_target_shape(DIR_PATHS["mswx"], "*.nc")
    
    # 1. Compute Reference (MSWX)
    print("  Computing MSWX Reference Climatology...")
    ds_mswx = compute_robust_mean_numpy(DIR_PATHS["mswx"], "*.nc", target_shape)
    if ds_mswx is None:
        print("CRITICAL: MSWX data not found.")
        return

    # Use the UNCLIPPED MSWX coordinates as the master grid (Size: 500x500)
    master_lat = ds_mswx['tas'].lat
    master_lon = ds_mswx['tas'].lon
    master_coords = {"lat": master_lat, "lon": master_lon}

    # Prepare Reference Clip for comparison
    try:
        # Assign coords explicitly to ensure consistency
        mswx_da = ds_mswx['tas'].assign_coords(master_coords)
        mswx_clip = safe_clip(mswx_da, india_gdf)
    except Exception as e:
        print(f"CRITICAL: Clipping MSWX failed: {e}")
        return

    metrics_registry = []

    # 2. Iterate Models
    for model in MODELS:
        print(f"  Processing: {model}")
        
        # Load Means (These return 500x500 grids)
        ds_raw = compute_robust_mean_numpy(DIR_PATHS["raw"], f"*{model}.nc", target_shape)
        ds_qdm = compute_robust_mean_numpy(DIR_PATHS["qdm"], f"*{model}.nc", target_shape)
        ds_ddpm = compute_robust_mean_numpy(DIR_PATHS["ddpm"], f"*{model}.nc", target_shape)
        
        if any(x is None for x in [ds_raw, ds_qdm, ds_ddpm]):
            print(f"    Skipping {model} (missing data)")
            continue

        # FLIP DDPM (Crucial Step from original code)
        ds_ddpm['tas'].values = np.flip(ds_ddpm['tas'].values, axis=0)

        try:
            # Assign the UNCLIPPED MASTER COORDINATES to all datasets
            # This ensures they are all 500x500 matching the original MSWX grid
            raw_da = ds_raw['tas'].assign_coords(master_coords)
            qdm_da = ds_qdm['tas'].assign_coords(master_coords)
            ddpm_da = ds_ddpm['tas'].assign_coords(master_coords)
            
            # Now Clip (500x500 -> ~300x300)
            raw_clip = safe_clip(raw_da, india_gdf)
            qdm_clip = safe_clip(qdm_da, india_gdf)
            ddpm_clip = safe_clip(ddpm_da, india_gdf)
            
            # Align everything to MSWX clip (Intersection) to remove any edge pixels mismatch
            raw_clip, mswx_aligned = xr.align(raw_clip, mswx_clip, join='inner')
            qdm_clip, _ = xr.align(qdm_clip, mswx_aligned, join='inner')
            ddpm_clip, _ = xr.align(ddpm_clip, mswx_aligned, join='inner')

            # Calculate Stats
            metrics_registry.append(calculate_spatial_metrics(model, "Raw", raw_clip - mswx_aligned))
            metrics_registry.append(calculate_spatial_metrics(model, "QDM", qdm_clip - mswx_aligned))
            metrics_registry.append(calculate_spatial_metrics(model, "DDPM", ddpm_clip - mswx_aligned))
            
        except Exception as e:
            print(f"    Error processing {model}: {e}")
            continue

    # ==========================================
    # 4. OUTPUT GENERATION
    # ==========================================
    print("\n" + "="*80)
    print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
    print("="*80 + "\n")
    
    print("**Model Bias Comparison: Raw vs QDM vs DDPM**")
    print("Analysis of spatial bias reduction across 10 CMIP6 models.")
    
    df = pd.DataFrame(metrics_registry)
    
    if df.empty:
        print("No metrics generated.")
    else:
        # Table 1: Detailed Metrics by Model & Stage
        print("\n### 1. Detailed Bias Metrics (All Models)")
        df_sort = df.sort_values(['Model', 'RMSE'])
        print(df_sort[['Model', 'Stage', 'Mean Bias (MB)', 'RMSE', 'Std Dev (SD)']].round(3).to_markdown(index=False))

        # Table 2: Improvement Summary
        print("\n### 2. Aggregated Improvement Analysis")
        # Group by Stage to get average performance across all 10 models
        df_agg = df.groupby('Stage')[['Mean Bias (MB)', 'RMSE', 'Std Dev (SD)']].mean().reindex(['Raw', 'QDM', 'DDPM'])
        print(df_agg.round(3).to_markdown())
        
        # Calculate Improvement % of DDPM over Raw and QDM (using RMSE)
        try:
            raw_rmse = df_agg.loc['Raw', 'RMSE']
            qdm_rmse = df_agg.loc['QDM', 'RMSE']
            ddpm_rmse = df_agg.loc['DDPM', 'RMSE']
            
            imp_raw = ((raw_rmse - ddpm_rmse) / raw_rmse) * 100
            imp_qdm = ((qdm_rmse - ddpm_rmse) / qdm_rmse) * 100
            
            print(f"\n- **DDPM Improvement over Raw (RMSE):** {imp_raw:.2f}%")
            print(f"- **DDPM Improvement over QDM (RMSE):** {imp_qdm:.2f}%")
        except:
            print("Could not calculate improvement percentages.")

    print("\n" + "="*80)
    print("END OF OUTPUT")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# STATISTICAL EXTRACTION: Sobel Gradient (Texture) Analysis
# Quantifies "Roughness" and "Structural Detail" without visual plots.

import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
from scipy import ndimage
import os
import glob
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODELS = [
    "CESM2", "CESM2-WACCM", "EC-Earth3-AerChem", "EC-Earth3-CC", 
    "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "INM-CM4-8", "INM-CM5-0", 
    "MPI-ESM1-2-HR", "NorESM2-MM"
]

SHAPEFILE_PATH = "<SET YOUR PATH HERE>"
DIR_PATHS = {
    "raw": "<SET YOUR PATH HERE>",
    "qdm": "<SET YOUR PATH HERE>",
    "ddpm": "<SET YOUR PATH HERE>",
    "mswx": "<SET YOUR PATH HERE>"
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_target_shape(folder_path, pattern="*.nc"):
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                if data.ndim == 3: return data.shape[1:]
                elif data.ndim == 2: return data.shape
        except: continue
    return None

def compute_robust_mean_numpy(folder_path, pattern, expected_shape_2d):
    """Computes time-mean map (climatology)."""
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None

    running_sum = np.zeros(expected_shape_2d, dtype=np.float64)
    total_count = np.zeros(expected_shape_2d, dtype=np.int32)
    valid_count = 0
    ref_ds = None

    for f in tqdm(files, desc=f"  Loading {pattern}", leave=False):
        try:
            if os.path.getsize(f) < 10240: continue
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                if data.ndim == 3:
                    if data.shape[1:] != expected_shape_2d: continue
                    val = data.sum(dim='time').values
                    cnt = data.notnull().sum(dim='time').values
                elif data.ndim == 2:
                    if data.shape != expected_shape_2d: continue
                    val = data.values
                    cnt = data.notnull().values.astype(int)
                else: continue
                running_sum += np.nan_to_num(val)
                total_count += cnt
                valid_count += 1
                if ref_ds is None:
                    ref_ds = ds.copy(deep=True)
                    if 'time' in ref_ds.dims: ref_ds = ref_ds.isel(time=0, drop=True)
        except: continue

    if valid_count == 0 or ref_ds is None: return None
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_arr = running_sum / total_count
    return xr.Dataset({'tas': (('lat', 'lon'), mean_arr)}, coords={'lat': ref_ds['lat'], 'lon': ref_ds['lon']})

def safe_clip(da, gdf):
    """Robust clipping ensuring CRS consistency."""
    if 'latitude' in da.coords: da = da.rename({'latitude': 'lat'})
    if 'longitude' in da.coords: da = da.rename({'longitude': 'lon'})
    da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)
    return da.rio.clip(gdf.geometry, gdf.crs, drop=True)

def compute_gradient_magnitude(da):
    """Calculates Magnitude of gradients (Texture Strength)."""
    filled = da.fillna(0).values
    sy = ndimage.sobel(filled, axis=0)
    sx = ndimage.sobel(filled, axis=1)
    mag = np.hypot(sx, sy)
    mask = da.notnull().values
    mag = np.where(mask, mag, np.nan)
    return xr.DataArray(mag, coords=da.coords)

def calculate_texture_metrics(model_name, stage_name, grad_model, grad_ref):
    """Computes stats comparing Model Texture vs Reference Texture."""
    
    # Flatten and remove NaNs
    g_mod = grad_model.values.flatten()
    g_mod = g_mod[np.isfinite(g_mod)]
    
    g_ref = grad_ref.values.flatten()
    g_ref = g_ref[np.isfinite(g_ref)]
    
    if len(g_mod) == 0 or len(g_ref) == 0:
        return None
    
    # 1. Mean Magnitude (How rough is it?)
    mean_mod = np.mean(g_mod)
    mean_ref = np.mean(g_ref)
    
    # 2. Roughness Ratio (1.0 = Ideal)
    # < 1.0 means too smooth (blurred)
    # > 1.0 means too noisy
    roughness_ratio = mean_mod / mean_ref if mean_ref > 0 else np.nan
    
    # 3. RMSE of Texture (Pixel-wise error in gradients)
    # We need aligned arrays for RMSE, so use the xarray objects before flattening
    diff = grad_model - grad_ref
    diff_vals = diff.values.flatten()
    diff_vals = diff_vals[np.isfinite(diff_vals)]
    rmse_texture = np.sqrt(np.mean(diff_vals**2))
    
    # 4. Correlation of Texture (Are edges in the right place?)
    # Align vectors
    if len(g_mod) == len(g_ref):
        corr = np.corrcoef(g_mod, g_ref)[0, 1]
    else:
        # Fallback if masking differed slightly (rare with align)
        corr = np.nan

    return {
        'Model': model_name,
        'Stage': stage_name,
        'Mean Gradient': mean_mod,
        'Roughness Ratio (Ideal=1)': roughness_ratio,
        'Texture RMSE': rmse_texture,
        'Texture Correlation': corr
    }

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def main():
    print("Initializing Sobel Texture Analysis...")
    
    try:
        india_gdf = gpd.read_file(SHAPEFILE_PATH).to_crs("EPSG:4326")
    except Exception as e:
        print(f"CRITICAL: Shapefile error: {e}")
        return

    target_shape = get_target_shape(DIR_PATHS["mswx"], "*.nc")
    
    # 1. MSWX Reference
    print("  Computing MSWX Gradient Field...")
    ds_mswx = compute_robust_mean_numpy(DIR_PATHS["mswx"], "*.nc", target_shape)
    if ds_mswx is None: return

    # Master Coords & Clip
    master_coords = {"lat": ds_mswx['tas'].lat, "lon": ds_mswx['tas'].lon}
    mswx_da = ds_mswx['tas'].assign_coords(master_coords)
    try:
        mswx_clip = safe_clip(mswx_da, india_gdf)
        grad_mswx = compute_gradient_magnitude(mswx_clip)
    except Exception as e:
        print(f"  Clipping error: {e}")
        return

    metrics_registry = []

    # 2. Iterate Models
    for model in MODELS:
        print(f"  Processing: {model}")
        
        # Load
        ds_raw = compute_robust_mean_numpy(DIR_PATHS["raw"], f"*{model}.nc", target_shape)
        ds_qdm = compute_robust_mean_numpy(DIR_PATHS["qdm"], f"*{model}.nc", target_shape)
        ds_ddpm = compute_robust_mean_numpy(DIR_PATHS["ddpm"], f"*{model}.nc", target_shape)
        
        if any(x is None for x in [ds_raw, ds_qdm, ds_ddpm]): continue

        # Flip DDPM
        ds_ddpm['tas'].values = np.flip(ds_ddpm['tas'].values, axis=0)

        try:
            # Assign Unclipped Master Coords
            raw_da = ds_raw['tas'].assign_coords(master_coords)
            qdm_da = ds_qdm['tas'].assign_coords(master_coords)
            ddpm_da = ds_ddpm['tas'].assign_coords(master_coords)
            
            # Clip
            raw_clip = safe_clip(raw_da, india_gdf)
            qdm_clip = safe_clip(qdm_da, india_gdf)
            ddpm_clip = safe_clip(ddpm_da, india_gdf)
            
            # Align
            raw_clip, mswx_aligned = xr.align(raw_clip, mswx_clip, join='inner')
            qdm_clip, _ = xr.align(qdm_clip, mswx_aligned, join='inner')
            ddpm_clip, _ = xr.align(ddpm_clip, mswx_aligned, join='inner')
            
            # Compute Gradients
            grad_raw = compute_gradient_magnitude(raw_clip)
            grad_qdm = compute_gradient_magnitude(qdm_clip)
            grad_ddpm = compute_gradient_magnitude(ddpm_clip)
            
            # Re-align gradients to be safe
            grad_raw, grad_mswx_a = xr.align(grad_raw, grad_mswx, join='inner')
            grad_qdm, _ = xr.align(grad_qdm, grad_mswx_a, join='inner')
            grad_ddpm, _ = xr.align(grad_ddpm, grad_mswx_a, join='inner')

            # Calculate Metrics
            metrics_registry.append(calculate_texture_metrics(model, "Raw", grad_raw, grad_mswx_a))
            metrics_registry.append(calculate_texture_metrics(model, "QDM", grad_qdm, grad_mswx_a))
            metrics_registry.append(calculate_texture_metrics(model, "DDPM", grad_ddpm, grad_mswx_a))
            
        except Exception as e:
            print(f"    Error {model}: {e}")
            continue

    # ==========================================
    # 4. OUTPUT GENERATION
    # ==========================================
    print("\n" + "="*80)
    print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
    print("="*80 + "\n")
    
    print("**Sobel Texture Analysis: Gradient Fidelity**")
    print("Quantification of high-frequency detail and spatial structure.")
    
    df = pd.DataFrame(metrics_registry)
    
    if not df.empty:
        # Table 1: Detailed by Model
        print("\n### 1. Detailed Texture Metrics")
        print(df.sort_values(['Model', 'Stage'])[['Model', 'Stage', 'Mean Gradient', 'Roughness Ratio (Ideal=1)', 'Texture Correlation']].round(3).to_markdown(index=False))

        # Table 2: Aggregated Summary
        print("\n### 2. Aggregated Texture Performance")
        df_agg = df.groupby('Stage')[['Roughness Ratio (Ideal=1)', 'Texture RMSE', 'Texture Correlation']].mean().reindex(['Raw', 'QDM', 'DDPM'])
        print(df_agg.round(3).to_markdown())
        
        # Key Stat Calculation
        raw_rough = df_agg.loc['Raw', 'Roughness Ratio (Ideal=1)']
        ddpm_rough = df_agg.loc['DDPM', 'Roughness Ratio (Ideal=1)']
        print(f"\n- **Raw GCM Roughness:** {raw_rough:.2f} (Indicating smoothing/blur)")
        print(f"- **DDPM Roughness:** {ddpm_rough:.2f} (Indicating realistic detail recovery)")

    print("\n" + "="*80)
    print("END OF OUTPUT")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# STATISTICAL EXTRACTION: Power Spectral Density (PSD) Analysis
# Quantifies Effective Resolution and Spectral Fidelity.

import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import os
import glob
from scipy import fftpack, stats
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODELS = [
    "CESM2", "CESM2-WACCM", "EC-Earth3-AerChem", "EC-Earth3-CC", 
    "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "INM-CM4-8", "INM-CM5-0", 
    "MPI-ESM1-2-HR", "NorESM2-MM"
]

PIXEL_SIZE_KM = 11.1
SHAPEFILE_PATH = "<SET YOUR PATH HERE>"
DIR_PATHS = {
    "raw": "<SET YOUR PATH HERE>",
    "qdm": "<SET YOUR PATH HERE>",
    "ddpm": "<SET YOUR PATH HERE>",
    "mswx": "<SET YOUR PATH HERE>"
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_target_shape(folder_path, pattern="*.nc"):
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                if data.ndim == 3: return data.shape[1:]
                elif data.ndim == 2: return data.shape
        except: continue
    return None

def compute_robust_mean_numpy(folder_path, pattern, expected_shape_2d):
    """Calculates temporal mean (climatology)."""
    search_path = os.path.join(folder_path, pattern)
    files = sorted(glob.glob(search_path))
    if not files: return None

    running_sum = np.zeros(expected_shape_2d, dtype=np.float64)
    total_count = np.zeros(expected_shape_2d, dtype=np.int32)
    valid_count = 0
    ref_ds = None

    for f in tqdm(files, desc=f"  Loading {pattern}", leave=False):
        try:
            if os.path.getsize(f) < 10240: continue
            with xr.open_dataset(f) as ds:
                var = 'tas' if 'tas' in ds else 'air_temperature'
                data = ds[var]
                if data.ndim == 3:
                    if data.shape[1:] != expected_shape_2d: continue
                    val = data.sum(dim='time').values
                    cnt = data.notnull().sum(dim='time').values
                elif data.ndim == 2:
                    if data.shape != expected_shape_2d: continue
                    val = data.values
                    cnt = data.notnull().values.astype(int)
                else: continue
                running_sum += np.nan_to_num(val)
                total_count += cnt
                valid_count += 1
                if ref_ds is None:
                    ref_ds = ds.copy(deep=True)
                    if 'time' in ref_ds.dims: ref_ds = ref_ds.isel(time=0, drop=True)
        except: continue

    if valid_count == 0 or ref_ds is None: return None
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_arr = running_sum / total_count
    return xr.Dataset({'tas': (('lat', 'lon'), mean_arr)}, coords={'lat': ref_ds['lat'], 'lon': ref_ds['lon']})

def compute_rapsd(data_array):
    """Computes Radially Averaged Power Spectral Density (RAPSD)."""
    if 0 in data_array.shape: return None, None
    if np.isnan(data_array).any():
        data_filled = np.nan_to_num(data_array, nan=np.nanmean(data_array))
    else:
        data_filled = data_array

    data_filled = data_filled - np.mean(data_filled)
    F = fftpack.fft2(data_filled)
    F_shift = fftpack.fftshift(F)
    P = np.abs(F_shift)**2
    
    H, W = P.shape
    center = (H // 2, W // 2)
    y, x = np.indices((H, W))
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)

    tbin = np.bincount(r.ravel(), P.ravel())
    nr = np.bincount(r.ravel())
    radial_profile = tbin / nr
    
    # Frequency in km^-1
    freq_pixel = np.linspace(0, 0.5, len(radial_profile)) 
    freq_km = freq_pixel / PIXEL_SIZE_KM
    return freq_km, radial_profile

def calculate_spectral_metrics(model_name, stage_name, freq, psd, psd_ref):
    """Calculates Slope, Effective Resolution, and High-Freq Energy Ratio."""
    
    # 1. Spectral Slope (Log-Log Regression on mid-high freqs > 0.01 km^-1)
    valid = (freq > 0.01) & (psd > 0)
    if np.sum(valid) > 5:
        slope, _, _, _, _ = stats.linregress(np.log10(freq[valid]), np.log10(psd[valid]))
    else:
        slope = np.nan

    # 2. Effective Resolution (Scale where Model Power < 50% Ref Power)
    ratio = psd / psd_ref
    eff_res_km = np.nan
    
    try:
        drop_idx = np.where(ratio[valid] < 0.5)[0]
        if len(drop_idx) > 0:
            k_cutoff = freq[valid][drop_idx[0]]
            eff_res_km = 1 / k_cutoff
        else:
            # If model strictly adheres to ref (never drops below 0.5), use Nyquist
            max_freq = freq[-1]
            eff_res_km = 1 / max_freq 
    except: pass
    
    # 3. High Frequency Energy Ratio (Are we resolving small scales?)
    # Average ratio for wavelengths < 50km (freq > 0.02)
    high_freq_mask = freq > 0.02
    if np.sum(high_freq_mask) > 0:
        high_freq_ratio = np.mean(ratio[high_freq_mask])
    else:
        high_freq_ratio = np.nan

    return {
        'Model': model_name,
        'Stage': stage_name,
        'Spectral Slope': slope,
        'Effective Resolution (km)': eff_res_km,
        'High-Freq Energy Ratio': high_freq_ratio
    }

def safe_clip(da, gdf):
    if 'latitude' in da.coords: da = da.rename({'latitude': 'lat'})
    if 'longitude' in da.coords: da = da.rename({'longitude': 'lon'})
    da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)
    return da.rio.clip(gdf.geometry, gdf.crs, drop=True)

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def main():
    print("Initializing Spectral Analysis...")
    
    try:
        india_gdf = gpd.read_file(SHAPEFILE_PATH).to_crs("EPSG:4326")
    except Exception as e:
        print(f"CRITICAL: Shapefile error: {e}")
        return

    target_shape = get_target_shape(DIR_PATHS["mswx"], "*.nc")
    
    # 1. MSWX Reference
    print("  Computing MSWX Reference PSD...")
    ds_mswx = compute_robust_mean_numpy(DIR_PATHS["mswx"], "*.nc", target_shape)
    if ds_mswx is None: return

    # Master Coords & Clip
    master_coords = {"lat": ds_mswx['tas'].lat, "lon": ds_mswx['tas'].lon}
    mswx_da = ds_mswx['tas'].assign_coords(master_coords)
    
    try:
        # Robust Crop Logic (Handle Lon Wrapping)
        minx, miny, maxx, maxy = india_gdf.total_bounds
        lat_slice = slice(maxy, miny) if mswx_da.lat[0] > mswx_da.lat[-1] else slice(miny, maxy)
        mswx_box = mswx_da.sel(lon=slice(minx, maxx), lat=lat_slice)
        
        freq_mswx, psd_mswx = compute_rapsd(mswx_box.values)
        
        # Calculate Reference Slope
        ref_metrics = calculate_spectral_metrics("Reference", "MSWX", freq_mswx, psd_mswx, psd_mswx)
        ref_slope = ref_metrics['Spectral Slope']
        
    except Exception as e:
        print(f"  Reference processing error: {e}")
        return

    metrics_registry = []

    # 2. Iterate Models
    for model in MODELS:
        print(f"  Processing: {model}")
        
        ds_raw = compute_robust_mean_numpy(DIR_PATHS["raw"], f"*{model}.nc", target_shape)
        ds_qdm = compute_robust_mean_numpy(DIR_PATHS["qdm"], f"*{model}.nc", target_shape)
        ds_ddpm = compute_robust_mean_numpy(DIR_PATHS["ddpm"], f"*{model}.nc", target_shape)
        
        if any(x is None for x in [ds_raw, ds_qdm, ds_ddpm]): continue

        ds_ddpm['tas'].values = np.flip(ds_ddpm['tas'].values, axis=0)

        try:
            # Assign Coords
            raw_da = ds_raw['tas'].assign_coords(master_coords)
            qdm_da = ds_qdm['tas'].assign_coords(master_coords)
            ddpm_da = ds_ddpm['tas'].assign_coords(master_coords)
            
            # Box Select (Same as Reference)
            raw_box = raw_da.sel(lon=slice(minx, maxx), lat=lat_slice)
            qdm_box = qdm_da.sel(lon=slice(minx, maxx), lat=lat_slice)
            ddpm_box = ddpm_da.sel(lon=slice(minx, maxx), lat=lat_slice)
            
            # Compute PSD
            f_r, p_r = compute_rapsd(raw_box.values)
            f_q, p_q = compute_rapsd(qdm_box.values)
            f_d, p_d = compute_rapsd(ddpm_box.values)
            
            # Metrics
            metrics_registry.append(calculate_spectral_metrics(model, "Raw", f_r, p_r, psd_mswx))
            metrics_registry.append(calculate_spectral_metrics(model, "QDM", f_q, p_q, psd_mswx))
            metrics_registry.append(calculate_spectral_metrics(model, "DDPM", f_d, p_d, psd_mswx))
            
        except Exception as e:
            print(f"    Error {model}: {e}")
            continue

    # ==========================================
    # 4. OUTPUT GENERATION
    # ==========================================
    print("\n" + "="*80)
    print("OUTPUT FOR AI PROMPT (COPY BELOW THIS LINE)")
    print("="*80 + "\n")
    
    print("**Power Spectral Density (PSD) Analysis: Effective Resolution**")
    print(f"Reference (MSWX) Spectral Slope: {ref_slope:.3f}")
    
    df = pd.DataFrame(metrics_registry)
    
    if not df.empty:
        # Table 1: Detailed by Model
        print("\n### 1. Detailed Spectral Metrics")
        print(df.sort_values(['Model', 'Stage'])[['Model', 'Stage', 'Spectral Slope', 'Effective Resolution (km)', 'High-Freq Energy Ratio']].round(3).to_markdown(index=False))

        # Table 2: Aggregated Summary
        print("\n### 2. Aggregated Spectral Performance")
        df_agg = df.groupby('Stage')[['Spectral Slope', 'Effective Resolution (km)', 'High-Freq Energy Ratio']].mean().reindex(['Raw', 'QDM', 'DDPM'])
        print(df_agg.round(3).to_markdown())
        
        # Improvement Stats
        raw_res = df_agg.loc['Raw', 'Effective Resolution (km)']
        ddpm_res = df_agg.loc['DDPM', 'Effective Resolution (km)']
        print(f"\n- **Raw GCM Effective Resolution:** {raw_res:.1f} km (Coarse)")
        print(f"- **DDPM Effective Resolution:** {ddpm_res:.1f} km (Fine-scale resolved)")
        
        ratio_raw = df_agg.loc['Raw', 'High-Freq Energy Ratio']
        ratio_ddpm = df_agg.loc['DDPM', 'High-Freq Energy Ratio']
        print(f"- **Raw Energy Ratio (<50km):** {ratio_raw:.2f} (Missing energy)")
        print(f"- **DDPM Energy Ratio (<50km):** {ratio_ddpm:.2f} (Recovered energy)")

    print("\n" + "="*80)
    print("END OF OUTPUT")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr
import numpy as np
import os

# List of file paths
file_paths = [
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>",
    "<SET YOUR PATH HERE>"
]

print(f"{'Model Name':<25} | {'Lat Res (°)':<12} | {'Lon Res (°)':<12}")
print("-" * 75)

for fpath in file_paths:
    if not os.path.exists(fpath):
        print(f"File not found: {os.path.basename(fpath)}")
        continue

    try:
        # Extract Model Name
        filename = os.path.basename(fpath)
        model_name = filename.split('_')[2]

        # Open dataset (decode_times=False is faster)
        with xr.open_dataset(fpath, decode_times=False) as ds:
            
            # Find Lat/Lon variables
            lat_name = next((d for d in ds.coords if 'lat' in d.lower()), None)
            lon_name = next((d for d in ds.coords if 'lon' in d.lower()), None)

            if lat_name and lon_name:
                lats = ds[lat_name].values
                lons = ds[lon_name].values

                # Calculate Resolution (Difference between adjacent points)
                # We take the median to handle Gaussian grids where spacing varies slightly
                lat_res = np.median(np.diff(lats))
                lon_res = np.median(np.diff(lons))
                
                # Check if Gaussian (non-uniform lat spacing)
                is_gaussian = not np.allclose(np.diff(lats), lat_res, atol=1e-4)
                

                print(f"{model_name:<25} | {abs(lat_res):<12.4f} | {abs(lon_res):<12.4f}")
            else:
                print(f"{model_name:<25} | {'?':<12} | {'?':<12} | Unknown Coords")

    except Exception as e:
        print(f"Error reading {os.path.basename(fpath)}: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Q1 Journal Formatting: Font Settings ---
# This forces the text to use Arial, which is the standard for academic figures.
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = 12       # Base font size
plt.rcParams['axes.labelsize'] = 12  # Axis labels
plt.rcParams['axes.titlesize'] = 14  # Subplot titles
plt.rcParams['legend.fontsize'] = 10 # Legend

# --- Parameters from Methodology ---
P = 128                # Patch size
stride = 64            # Stride (50% overlap)
sigma_factor = 0.3     # Gaussian spread factor
sigma_g = sigma_factor * P

def gaussian_kernel(size, sigma):
    """Generates the 2D Gaussian Kernel (Eq 12)."""
    ax = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel

def plot_stitching_schematic():
    # Setup Figure
    fig = plt.figure(figsize=(13, 5)) # Slightly wider for better spacing
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.4])
    
    # --- Panel A: 2D Gaussian Kernel Heatmap ---
    ax1 = plt.subplot(gs[0])
    kernel = gaussian_kernel(P, sigma_g)
    
    im = ax1.imshow(kernel, extent=[0, P, 0, P], origin='lower', cmap='viridis')
    ax1.set_title(r'(a) 2D Weighting Kernel $w_k$', fontweight='bold')
    ax1.set_xlabel(r'Pixel $i$')
    ax1.set_ylabel(r'Pixel $j$')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.set_label('Weight magnitude')
    
    # --- Panel B: 1D Cross-Section Stitching Demo ---
    ax2 = plt.subplot(gs[1])
    
    # Create 1D slices for two overlapping patches
    x = np.arange(0, P + stride) # Domain covering 2 patches
    
    # Gaussian 1 (Left Patch)
    g1_center = P / 2
    g1 = np.exp(-(x - g1_center)**2 / (2 * sigma_g**2))
    g1[x >= P] = 0 # Mask outside patch 1 definition
    
    # Gaussian 2 (Right Patch - shifted by stride)
    g2_center = P / 2 + stride
    g2 = np.exp(-(x - g2_center)**2 / (2 * sigma_g**2))
    g2[x < stride] = 0 # Mask outside patch 2 definition
    
    # Plotting Lines
    ax2.plot(x, g1, 'b--', linewidth=2.5, label=r'Patch 1 Weight ($w_1$)')
    ax2.plot(x, g2, 'r--', linewidth=2.5, label=r'Patch 2 Weight ($w_2$)')
    
    # Shading the Regions
    # 1. Left (Patch 1 Only)
    ax2.fill_between(x, 0, g1, where=(x < stride), color='blue', alpha=0.1)
    # 2. Overlap (Mixing) - This is the key part
    ax2.fill_between(x, 0, np.maximum(g1, g2), where=((x >= stride) & (x <= P)), color='purple', alpha=0.2)
    # 3. Right (Patch 2 Only)
    ax2.fill_between(x, 0, g2, where=(x > P), color='red', alpha=0.1)
    
    # Vertical Dividers
    ax2.axvline(stride, color='gray', linestyle=':', alpha=0.8, linewidth=1.5)
    ax2.axvline(P, color='gray', linestyle=':', alpha=0.8, linewidth=1.5)
    
    # Annotations (Bold for clarity)
    ax2.text(stride/2, 0.5, 'Patch 1 Only', ha='center', fontweight='bold', color='dimgray')
    ax2.text(stride + (P-stride)/2, 0.2, 'Overlap Region\n(Mixing)', ha='center', fontweight='bold', color='black')
    ax2.text(P + (stride)/2, 0.5, 'Patch 2 Only', ha='center', fontweight='bold', color='dimgray')

    ax2.set_title(r'(b) 1D Cross-Section of Overlap Aggregation', fontweight='bold')
    ax2.set_xlabel('Spatial Coordinate (pixels)')
    ax2.set_ylabel('Weight Contribution')
    ax2.legend(loc='upper right', framealpha=0.9)
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    
    # # --- SAVE COMMAND FOR MANUSCRIPT ---
    # # Save as PDF (Vector) for maximum quality
    # plt.savefig('Gaussian_Stitching_Schematic.pdf', dpi=300, bbox_inches='tight')
    # # Also save as PNG for quick viewing
    plt.savefig("<SET YOUR PATH HERE>", dpi=300, bbox_inches='tight')
    
    plt.show()

if __name__ == "__main__":
    plot_stitching_schematic()

In [ ]:
import xarray as xr

ds1 = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds1)